# Assignment 1: Generalization

*Authors:* Thomas Adler, Eric Volkmann, Markus Holzleitner

*Copyright statement:* This material, no matter whether in printed or electronic form, may be used for personal and non-commercial educational use only. Any reproduction of this manuscript, no matter whether as a whole or in parts, no matter whether in printed or in electronic form, requires explicit prior acceptance of the authors.

## Preliminaries

* Answer theory exercises in the markdown cells and programming exercises in the code cells marked `YOUR SOLUTION HERE`. Informal arguments and sketches are sufficient unless a calculation is asked for.
* Packages: `numpy`, `scipy`, `matplotlib`, `torch` (CPU is sufficient). The experiment in Exercise 6 runs for about 3 minutes.

**Notation:**

| Symbol | Meaning |
|---|---|
| $\mathbf{x} \in X \subseteq \mathbb{R}^d$, $y \in \{-1, 1\}$ | input vector and label, drawn from the unknown distribution $p(\mathbf{x}, y)$ |
| $\mathbf{Z} = \big((\mathbf{x}_1, y_1), \dots, (\mathbf{x}_l, y_l)\big) \sim p^l$ | i.i.d. training set of size $l$ |
| $f : X \to \{-1, 1\}$ | classifier; $\mathcal{F}$ is the hypothesis class we choose from |
| $L(y, y') = \mathbf{1}\{y \neq y'\}$ | zero-one loss |
| $R(f) = \mathbb{E}_{(\mathbf{x}, y) \sim p}[L(y, f(\mathbf{x}))]$ | expected risk = probability of an error on a new example |
| $R_{\mathrm{emp}}(f) = \frac{1}{l} \sum_{i=1}^{l} L(y_i, f(\mathbf{x}_i))$ | empirical risk = error rate on the training set |
| $f_{\mathcal{F}} \in \arg\min_{f \in \mathcal{F}} R(f)$ | best classifier in the class |
| $\hat f \in \arg\min_{f \in \mathcal{F}} R_{\mathrm{emp}}(f)$ | empirical risk minimiser (ERM), the classifier we actually learn |
| $\mathrm{VC}(\mathcal{F})$ | VC-dimension: the largest number $l$ of points such that some $l$ points can be labelled in all $2^l$ ways by functions from $\mathcal{F}$ ("shattered") |

Convention: $\mathrm{sgn}(z) = 1$ for $z \geq 0$ and $\mathrm{sgn}(z) = -1$ for $z < 0$. $\ln$ is the natural logarithm, $\log_2$ the binary logarithm.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

%matplotlib inline

## Concentration of the empirical risk (slides I.1)

For a fixed classifier $f$ the losses $X_i = L(y_i, f(\mathbf{x}_i))$, $i = 1, \dots, l$, are i.i.d. with $X_i \in \{0, 1\}$ and $\mathbb{P}(X_i = 1) = R(f)$, and $R_{\mathrm{emp}}(f) = \frac{1}{l} \sum_{i=1}^{l} X_i$ is their average.

### Exercise 1: Two bounds for the deviation of the training error (2 points)

a) (1 point) Show that $\mathbb{E}[R_{\mathrm{emp}}(f)] = R(f)$ and $\operatorname{Var}(R_{\mathrm{emp}}(f)) = \frac{R(f)\,(1 - R(f))}{l}$, and conclude with Chebyshev's inequality, $\mathbb{P}(\lvert X - \mathbb{E}[X] \rvert \geq \varepsilon) \leq \operatorname{Var}(X) / \varepsilon^2$, that
$$
    \mathbb{P}\big(\lvert R_{\mathrm{emp}}(f) - R(f) \rvert \geq \varepsilon\big) \leq \frac{1}{4 l \varepsilon^2}.
$$

########## YOUR SOLUTION HERE ##########

Consider a **fixed classifier** $f$ and independently sampled training examples. Let $r = R(f)$ denote its true error probability.

For each example, define the **error indicator**:

$$
X_i = L(y_i, f(\mathbf{x}_i)) \in \{0,1\}.
$$

Thus, $X_i$ is Bernoulli distributed with parameter $r$: it equals $1$ when the classifier makes an error and $0$ otherwise. Consequently,

$$
\mathbb{E}[X_i] = r,
\qquad
\operatorname{Var}(X_i) = r(1-r).
$$

The empirical risk is the average of these error indicators:

$$
R_{\mathrm{emp}}(f) = \frac{1}{l}\sum_{i=1}^{l}X_i.
$$

#### 1. Expected empirical risk

By linearity of expectation,

$$
\begin{aligned}
\mathbb{E}[R_{\mathrm{emp}}(f)]
&= \frac{1}{l}\sum_{i=1}^{l}\mathbb{E}[X_i] \\
&= \frac{lr}{l} \\
&= r \\
&= R(f).
\end{aligned}
$$

Therefore, the training error is an **unbiased estimate of the true error**: averaged over randomly sampled training sets, it equals the true risk.

#### 2. Variance of the empirical risk

Since the error indicators are independent, their variances add. Also, multiplying a random variable by $1/l$ multiplies its variance by $1/l^2$. Hence,

$$
\begin{aligned}
\operatorname{Var}(R_{\mathrm{emp}}(f))
&= \frac{1}{l^2}\sum_{i=1}^{l}\operatorname{Var}(X_i) \\
&= \frac{lr(1-r)}{l^2} \\
&= \frac{R(f)(1-R(f))}{l}.
\end{aligned}
$$

Thus,

$$
\boxed{
\operatorname{Var}(R_{\mathrm{emp}}(f))
= \frac{R(f)(1-R(f))}{l}
}
$$

This shows that the **variance decreases as the training set grows**.

#### 3. Apply Chebyshev's inequality

For any $\varepsilon > 0$,

$$
\begin{aligned}
\mathbb{P}\!\left(
\left|R_{\mathrm{emp}}(f)-R(f)\right| \geq \varepsilon
\right)
&\leq \frac{\operatorname{Var}(R_{\mathrm{emp}}(f))}{\varepsilon^2} \\
&= \frac{r(1-r)}{l\varepsilon^2}.
\end{aligned}
$$

For every $r \in [0,1]$,

$$
r(1-r)
= \frac{1}{4}-\left(r-\frac{1}{2}\right)^2
\leq \frac{1}{4}.
$$

Substituting this upper bound gives the **required result**:

$$
\boxed{
\mathbb{P}\!\left(
\left|R_{\mathrm{emp}}(f)-R(f)\right| \geq \varepsilon
\right)
\leq \frac{1}{4l\varepsilon^2}
}
$$

b) (1 point) Hoeffding's inequality (slide 26 of I.2) bounds the same probability by $2 e^{-2 l \varepsilon^2}$. Rewrite both bounds as statements of the form "with probability at least $1 - \delta$: $\lvert R_{\mathrm{emp}}(f) - R(f) \rvert < \varepsilon(l, \delta)$", fill in the table for $l = 1000$ and name the essential difference between the two bounds.

| $\delta$ | $\varepsilon$ from Chebyshev | $\varepsilon$ from Hoeffding |
|---|---|---|
| $0.1$ | | |
| $0.001$ | | |
| $10^{-6}$ | | |

########## YOUR SOLUTION HERE ##########

Let $\delta \in (0,1)$ be the allowed probability that the deviation reaches or exceeds our tolerance.

We choose $\varepsilon$ so that the upper bound on this probability equals $\delta$. Taking the complementary event then gives:

$$
\mathbb{P}\!\left(
\left|R_{\mathrm{emp}}(f)-R(f)\right|
< \varepsilon(l,\delta)
\right)
\geq 1-\delta.
$$

In other words, **with probability at least $1-\delta$, the empirical risk differs from the true risk by less than $\varepsilon(l,\delta)$**.

#### 1. Chebyshev's confidence bound

Set the bound from part (a) equal to $\delta$ and solve for $\varepsilon$:

$$
\begin{aligned}
\frac{1}{4l\varepsilon^2} &= \delta, \\
\varepsilon^2 &= \frac{1}{4l\delta}, \\
\varepsilon &= \frac{1}{2\sqrt{l\delta}}.
\end{aligned}
$$

Therefore,

$$
\boxed{
\varepsilon_{\mathrm{Cheb}}(l,\delta)
= \frac{1}{2\sqrt{l\delta}}
}
$$

Thus, **with probability at least $1-\delta$**,

$$
\left|R_{\mathrm{emp}}(f)-R(f)\right|
< \frac{1}{2\sqrt{l\delta}}.
$$

#### 2. Hoeffding's confidence bound

Starting from

$$
2e^{-2l\varepsilon^2} = \delta,
$$

divide by $2$ and take the natural logarithm:

$$
\begin{aligned}
e^{-2l\varepsilon^2} &= \frac{\delta}{2}, \\
-2l\varepsilon^2 &= \ln\!\left(\frac{\delta}{2}\right), \\
\varepsilon^2 &= \frac{\ln(2/\delta)}{2l}.
\end{aligned}
$$

Therefore,

$$
\boxed{
\varepsilon_{\mathrm{Hoef}}(l,\delta)
= \sqrt{\frac{\ln(2/\delta)}{2l}}
}
$$

Thus, **with probability at least $1-\delta$**,

$$
\left|R_{\mathrm{emp}}(f)-R(f)\right|
< \sqrt{\frac{\ln(2/\delta)}{2l}}.
$$

#### 3. Numerical values for $l=1000$

The tolerances below are absolute differences in error rates, rounded to six decimal places.

| $\delta$ | Confidence $1-\delta$ | $\varepsilon$ from Chebyshev | $\varepsilon$ from Hoeffding |
|---|---|---|---|
| $0.1$ | 90% | $0.050000$ | $0.038702$ |
| $0.001$ | 99.9% | $0.500000$ | $0.061648$ |
| $10^{-6}$ | 99.9999% | $15.811388$ | $0.085172$ |

For example, at **90% confidence**:

- Chebyshev guarantees a deviation below approximately **5 percentage points**.
- Hoeffding guarantees a deviation below approximately **3.87 percentage points**.

The guarantees use the exact expressions above; the table displays rounded approximations.

#### 4. Essential difference between the bounds

For a **fixed tolerance** $\varepsilon$:

- Chebyshev's probability bound decreases **polynomially** with the sample size, as $1/l$.
- Hoeffding's probability bound decreases **exponentially**, as $e^{-2l\varepsilon^2}$.

Chebyshev uses a variance bound. Hoeffding exploits the **independence and boundedness** of the losses, here $X_i \in [0,1]$.

When we increase the required confidence by decreasing $\delta$, the tolerances grow differently:

$$
\varepsilon_{\mathrm{Cheb}} \propto \delta^{-1/2},
\qquad
\varepsilon_{\mathrm{Hoef}} \propto \sqrt{\ln(2/\delta)}.
$$

This explains why **Hoeffding gives much tighter guarantees for the small values of $\delta$ in the table**.

For a fixed $\delta$, both tolerances decrease as $1/\sqrt{l}$.

#### 5. Interpretation of the largest Chebyshev tolerance

For $\delta = 10^{-6}$, Chebyshev's tolerance exceeds $1$.

Since both error rates lie in $[0,1]$, their absolute difference cannot exceed $1$. Therefore, this Chebyshev guarantee provides **no useful restriction**.

Hoeffding still yields a meaningful tolerance of approximately **8.52 percentage points**.

### Exercise 2: How tight are the bounds? (2 points, code)

Fix a classifier with $R(f) = 0.3$; then $l \cdot R_{\mathrm{emp}}(f) \sim \mathrm{Binomial}(l, 0.3)$, so the distribution of the training error can be simulated without knowing $p$ or $f$.

a) (1 point) Implement `tail_probability`, `chebyshev_bound` and `hoeffding_bound` in the next cell (see the docstrings).

b) (1 point) The cell after that plots the Monte-Carlo estimate of $\mathbb{P}(\lvert R_{\mathrm{emp}}(f) - R(f) \rvert \geq 0.05)$, both bounds and the normal approximation $2\,\Phi\big(-\varepsilon\sqrt{l / (R(f)(1 - R(f)))}\big)$ from the central limit theorem. Discuss: Which bound is better for small $l$, which for large $l$? By which factor do the bounds overestimate the actual probability at $l = 1000$? Which information about the distribution of the losses does each bound use, and which additional information could be used to tighten them? Why is the normal approximation not used in the theory, although it is much closer to the truth?

In [2]:
def tail_probability(l, R, eps, n_rep, rng):
    '''Monte-Carlo estimate of P(|R_emp(f) - R(f)| >= eps) for a fixed classifier f with risk R(f) = R,
    based on n_rep simulated training sets of size l. Hint: rng.binomial; compare with a small tolerance
    (>= eps - 1e-12) to avoid rounding artefacts at the boundary.'''
    ########## YOUR SOLUTION HERE ##########
    raise NotImplementedError("You have not implemented this function.")


def chebyshev_bound(l, R, eps):
    '''Chebyshev bound R (1 - R) / (l eps^2) from Exercise 1 (with the exact variance).'''
    ########## YOUR SOLUTION HERE ##########
    raise NotImplementedError("You have not implemented this function.")


def hoeffding_bound(l, eps):
    '''Hoeffding bound 2 exp(-2 l eps^2).'''
    ########## YOUR SOLUTION HERE ##########
    raise NotImplementedError("You have not implemented this function.")

In [3]:
# Nothing to implement here: simulation and plot.
R, eps, n_rep = 0.3, 0.05, 200_000
ls = np.array([10, 20, 50, 100, 200, 500, 1000])
rng = np.random.default_rng(0)

estimate = np.array([tail_probability(l, R, eps, n_rep, rng) for l in ls])
cheb = chebyshev_bound(ls, R, eps)
hoef = hoeffding_bound(ls, eps)
normal_approx = 2 * norm.cdf(-eps * np.sqrt(ls / (R * (1 - R))))

print(f"{'l':>6} {'Monte-Carlo':>12} {'Chebyshev':>10} {'Hoeffding':>10} {'normal approx.':>15}")
for row in zip(ls, estimate, cheb, hoef, normal_approx):
    print(f"{row[0]:6d} {row[1]:12.5f} {row[2]:10.4f} {row[3]:10.2e} {row[4]:15.2e}")

plt.figure(figsize=(6.5, 4.2))
plt.loglog(ls, estimate, "o-", label="Monte-Carlo estimate")
plt.loglog(ls, np.minimum(cheb, 1), "s--", label="Chebyshev bound")
plt.loglog(ls, np.minimum(hoef, 1), "^--", label="Hoeffding bound")
plt.loglog(ls, normal_approx, ":", label="normal approximation (CLT)")
plt.xlabel("sample size $l$")
plt.ylabel(r"$\mathbb{P}(|R_{\rm emp}(f) - R(f)| \geq \varepsilon)$")
plt.title(r"$R(f) = 0.3$, $\varepsilon = 0.05$")
plt.grid(alpha=0.3); plt.legend(); plt.show()

NotImplementedError: You have not implemented this function.

########## YOUR SOLUTION HERE ##########

## Finite hypothesis classes — how many examples per parameter? (slides I.2, 31–36)

For $|\mathcal{F}| = m$, slides 31 and 35 give, with probability at least $1 - \delta$,
$$
    R(\hat f) \;\leq\; R(f_{\mathcal{F}}) + 2 \sqrt{\frac{\ln(2m) + \ln(1/\delta)}{2l}},
    \qquad\text{i.e.}\qquad
    R(\hat f) - R(f_{\mathcal{F}}) \leq \varepsilon \quad \text{as soon as} \quad l \geq \frac{2\,\big(\ln(2m) + \ln(1/\delta)\big)}{\varepsilon^2}. \tag{1}
$$

### Exercise 3: Floating-point parameters and quantisation (3 points)

a) (1 point) A model with $k$ parameters, each stored with $b$ bits, has at most $m = 2^{kb}$ distinct parameter settings. Insert this into (1): how many training examples are required for $\varepsilon = \delta = 0.01$ and $b = 32$, as a function of $k$? Interpret the result.

########## YOUR SOLUTION HERE ##########

b) (1 point) Networks are often quantised to $b = 16$ or $b = 8$ bits. How does the number of examples *per parameter* change? Does this mean that quantised networks generalise better?

########## YOUR SOLUTION HERE ##########

In [ ]:
# Nothing to implement here: required sample size according to (1) for k parameters with b bits each.
def required_samples(k, b, eps=0.01, delta=0.01):
    m_log = k * b * np.log(2)                       # ln(m) for m = 2^(k b)
    return 2 * (m_log + np.log(2) + np.log(1 / delta)) / eps**2

ks = np.logspace(0, 4, 200)                     # 1 ... 10 000 parameters
plt.figure(figsize=(6.5, 4.2))
for b in [8, 16, 32]:
    plt.plot(ks, required_samples(ks, b), label=f"{b}-bit parameters")
for name, size, va in [("MNIST (60k)", 60_000, "bottom"), ("CIFAR-10 (50k)", 50_000, "top"), ("ImageNet (1.28M)", 1_280_000, "bottom")]:
    plt.axhline(size, color="gray", ls=":", lw=1)
    plt.text(ks[-1], size, name, ha="right", va=va, fontsize=8, color="gray")
plt.xscale("log"); plt.yscale("log"); plt.xlabel("number of parameters $k$"); plt.ylabel("required training set size $l$")
plt.title(r"bound (1) for $\varepsilon = \delta = 0.01$"); plt.legend(); plt.grid(alpha=0.3); plt.show()

c) (1 point) The cell above plots the required sample size for $b \in \{8, 16, 32\}$ together with the sizes of some standard data sets. How many parameters does the bound allow for ImageNet ($1.28$ million images) at $\varepsilon = \delta = 0.01$? What does this tell you about worst-case bounds and modern networks?

########## YOUR SOLUTION HERE ##########

## VC-dimension — capacity for infinite classes (slides I.2, 38–47)

For infinite classes the VC-dimension takes the role of $\ln m$ in (1) (slide 45). To show $\mathrm{VC}(\mathcal{F}) = n$: (i) exhibit *one* set of $n$ points on which *every* labelling is realised; (ii) show that for an *arbitrary* set of $n + 1$ points *some* labelling cannot be realised. Intervals and axis-aligned rectangles are treated in Section 1.4 of the derivations note.

### Exercise 4: VC-dimension of right triangles (3 points)

Consider the classifiers $f(\mathbf{x}) = +1$ if $\mathbf{x} \in T_{a,b,c}$ and $f(\mathbf{x}) = -1$ otherwise, where
$$
    T_{a, b, c} = \{ \mathbf{x} \in \mathbb{R}^2 : x_1 \geq a,\ x_2 \geq b,\ x_1 + x_2 \leq c \}
$$
is the right triangle with the right angle at $(a, b)$, legs parallel to the axes and hypotenuse on the line $x_1 + x_2 = c$ (empty for $c < a + b$). Show that $\mathrm{VC}(\mathcal{F}) = 3$.

a) (1 point) Give three points that are shattered. *Hint:* each side can cut off one point; choose the points such that each is the unique extreme point for one side (smallest $x_1$, smallest $x_2$, largest $x_1 + x_2$).

########## YOUR SOLUTION HERE ##########

b) (1 point) The four points $(1, 0), (0, 1), (-1, 0), (0, -1)$ are shattered by axis-aligned rectangles (derivations, Section 1.4). Which labelling can no triangle $T_{a,b,c}$ produce, and why?

########## YOUR SOLUTION HERE ##########

c) (1 point) Show that *no* four points can be shattered. *Hint:* generalise (b) — among any four points pick one with the smallest $x_1$, one with the smallest $x_2$ and one with the largest $x_1 + x_2$; what about the remaining point? (Same argument as for rectangles in derivations 1.4, with three "extreme directions" instead of four.)

########## YOUR SOLUTION HERE ##########

### Exercise 5: Lines, the XOR problem and a hidden layer (3 points)

Consider linear classifiers in the plane, $f(\mathbf{x}) = \mathrm{sgn}(w_1 x_1 + w_2 x_2 + b)$ (the perceptron of slide 11 of I.2 with a bias term): one side of a line is labelled $+1$, the other $-1$.

a) (1 point) Show that any three points not on a common line are shattered.

########## YOUR SOLUTION HERE ##########

b) (1 point) Show that no four points are shattered: consider (i) one point inside the triangle of the other three and (ii) the four corners of a convex quadrilateral with the labelling in which opposite corners share the label (the *XOR labelling*). Conclude $\mathrm{VC} = 3$. (Fact, no proof required: in $\mathbb{R}^d$ the VC-dimension of linear classifiers with bias is $d + 1$.)

########## YOUR SOLUTION HERE ##########

c) (1 point) *One hidden layer.* Consider a network with two hidden threshold units $h_j(\mathbf{x}) = \mathrm{sgn}(\mathbf{w}_j^\top \mathbf{x} + b_j)$, $j = 1, 2$, and output $f(\mathbf{x}) = \mathrm{sgn}(v_1 h_1(\mathbf{x}) + v_2 h_2(\mathbf{x}) + c)$. Give parameters that realise the XOR labelling $(+1, -1, +1, -1)$ of the points $(1, 0), (0, 1), (-1, 0), (0, -1)$, and explain why the network realises *all* $16$ labellings of these points ($14$ of them are realised by a single line — how can the network imitate a line?). What is your guess for the growth of the VC-dimension with the number $k$ of hidden units? (Exercise 6 measures it.)

########## YOUR SOLUTION HERE ##########

### Exercise 6: Measuring the VC-dimension of a small neural network (3 points, code)

Baum & Haussler (1989): a network of threshold units with $W$ weights and $N$ units has $\mathrm{VC} \leq 2 W \log_2(e N)$, and such networks can shatter a number of points proportional to $W$ — up to logarithms, the VC-dimension of a network is its number of parameters. We measure this for the network
$$
    f_\theta(\mathbf{x}) = \mathrm{sgn}\big(g_\theta(\mathbf{x})\big), \qquad g_\theta(\mathbf{x}) = \sum_{j=1}^{k} v_j \tanh(\mathbf{w}_j^\top \mathbf{x} + b_j) + c \qquad (\mathbf{x} \in \mathbb{R}^2,\ W = 4k + 1),
$$
with $\tanh$ instead of $\mathrm{sgn}$ inside so that it can be trained by gradient descent; for $k = 0$, $g_\theta(\mathbf{x}) = \mathbf{w}^\top \mathbf{x} + c$ is the linear classifier of Exercise 5 ($W = 3$).

*Protocol (code below).* Points: $l$ equally spaced points on the unit circle. Labellings: all $\mathbf{y}$ with $y_1 = +1$ (the rest follows by flipping the signs of $\mathbf{v}$ and $c$); at most $1024$, randomly chosen. Training: one network per labelling, all in parallel as a batch of $B$ networks, Adam on the squared hinge loss $\frac{1}{l}\sum_i \max(0, 1 - y_i g_\theta(\mathbf{x}_i))^2$, $5$ random restarts; a labelling is *realised* if at some point during training all $l$ points are classified correctly. $s(l, k)$ denotes the fraction of realised labellings and $\hat l(k)$ the largest $l$ with $s(l', k) = 1$ for all $l' \leq l$ — the experimental estimate of the VC-dimension. For $k \in \{0, 1, 2, 4, 8\}$ the loop stops once $s(l, k) < 0.5$.

a) (1 point) Implement `all_labelings` and `training_step` in the next cell (`forward` is given; see the docstrings) and run the test cell.

b) Run the experiment (about 3 minutes). The last cell prints $\hat l(k)$ together with the bound $2 W \log_2(eN)$, $N = k + 1$, and plots $s(l, k)$.

c) (2 points) Discuss: (1) $\hat l(0)$ and $\hat l(1)$ compared with Exercise 5 — why does one hidden unit not help? (2) How does $\hat l(k)$ grow with $k$ (or $W$)? Does the bound show the same trend, and is it tight? (3) Two reasons why $\hat l(k)$ can only be a *lower* estimate of the VC-dimension; which of the statements "these $l$ points are shattered" / "these $l$ points are not shattered" can such an experiment certify? (4) Slide 58 shows networks that fit *random* labels of their training set. What does this say about their VC-dimension relative to $l$, and about the bound on slide 45 in that regime?

In [ ]:
import torch
torch.manual_seed(0)


def circle_points(l):
    '''l equally spaced points on the unit circle, float tensor of shape (l, 2).'''
    phi = 2 * np.pi * np.arange(l) / l
    return torch.tensor(np.stack([np.cos(phi), np.sin(phi)], axis=1), dtype=torch.float32)


def init_params(B, k):
    '''Random initial parameters of B independent networks with k hidden units (k = 0: linear classifier).
    Shapes: W1 (B, k, 2), b1 (B, k), v (B, k), c (B,);  for k = 0: w (B, 2), c (B,).'''
    if k == 0:
        return [torch.randn(B, 2) * 0.5, torch.zeros(B)]                                  # w, c
    return [torch.randn(B, k, 2), torch.randn(B, k) * 0.5, torch.randn(B, k) / np.sqrt(k), torch.zeros(B)]   # W1, b1, v, c


def forward(params, X):
    '''Pre-sign outputs g_theta(x_i) of all B networks at once: X has shape (l, 2), the result has shape (B, l).'''
    if len(params) == 2:                                                   # k = 0: g(x) = w^T x + c
        w, c = params
        return w @ X.T + c[:, None]                                        # (B, 2) @ (2, l) -> (B, l)
    W1, b1, v, c = params
    H = torch.tanh(torch.einsum("bkj,lj->blk", W1, X) + b1[:, None, :])   # hidden activations, shape (B, l, k)
    return (H * v[:, None, :]).sum(-1) + c[:, None]                        # (B, l)


def all_labelings(l):
    '''All labellings y in {-1, +1}^l with y_1 = +1, as float tensor of shape (2**(l-1), l), each exactly once.
    Hint: binary representation of the numbers 0, ..., 2**(l-1) - 1.'''
    ########## YOUR SOLUTION HERE ##########
    raise NotImplementedError("You have not implemented this function.")


def training_step(params, opt, X, Y):
    '''One Adam step on the squared hinge loss for all B networks (Y has shape (B, l), forward(params, X) too).
    Returns a bool tensor of shape (B,): does network m currently classify all l points correctly?'''
    ########## YOUR SOLUTION HERE ##########
    raise NotImplementedError("You have not implemented this function.")


def realised_labelings(X, Y, k, steps=600, lr=0.05, restarts=5):
    '''Train one network per labelling (row of Y) and return a bool tensor of shape (B,):
    True if the labelling was realised at some point during training.'''
    realised = torch.zeros(Y.shape[0], dtype=torch.bool)
    for r in range(restarts):
        params = [p.requires_grad_() for p in init_params(Y.shape[0], k)]
        opt = torch.optim.Adam(params, lr=lr)
        for step in range(steps):
            realised |= training_step(params, opt, X, Y)
    return realised

In [ ]:
# Tests for (a). Nothing to implement here.
Y = all_labelings(4)
assert Y.shape == (8, 4), "wrong shape"
assert set(Y.unique().tolist()) == {-1.0, 1.0} and bool((Y[:, 0] == 1).all()), "entries must be +-1 with y_1 = +1"
assert len({tuple(row) for row in Y.tolist()}) == 8, "labellings must be distinct"

X = circle_points(3)
Y = all_labelings(3)                                   # 4 labellings of 3 points, all linearly realisable
params = [p.requires_grad_() for p in init_params(Y.shape[0], 0)]
opt = torch.optim.Adam(params, lr=0.05)
ok = training_step(params, opt, X, Y)
assert ok.shape == (4,) and ok.dtype == torch.bool, "training_step must return a bool tensor of shape (B,)"
for _ in range(300):
    ok |= training_step(params, opt, X, Y)
assert bool(ok.all()), "after 300 steps all 4 labellings of 3 points should be realised by a linear classifier"
print("All tests passed.")

In [ ]:
# Experiment for (b). Nothing to implement here (runtime: about 3 minutes on a laptop CPU).
import time

ks, l_max, max_labelings = [0, 1, 2, 4, 8], 16, 1024
results = {}                                    # (k, l) -> (number of realised labellings, number of tested labellings)

t_start = time.time()
for k in ks:
    for l in range(3, l_max + 1):
        Y = all_labelings(l)
        if Y.shape[0] > max_labelings:          # random subset of the labellings for large l
            perm = torch.randperm(Y.shape[0], generator=torch.Generator().manual_seed(l))
            Y = Y[perm[:max_labelings]]
        realised = realised_labelings(circle_points(l), Y, k)
        results[(k, l)] = (int(realised.sum()), Y.shape[0])
        print(f"k = {k}   l = {l:2d}   realised {int(realised.sum()):5d} of {Y.shape[0]:5d} tested labellings"
              f"   s(l, k) = {realised.float().mean():.3f}")
        if realised.float().mean() < 0.5:       # far beyond the VC-dimension: stop
            break
print(f"total time: {time.time() - t_start:.0f} s")

In [ ]:
# Summary table and plot for (b). Nothing to implement here.
print(f"{'k':>3} {'W':>4} {'l_hat(k)':>10} {'2 W log2(e N)':>15}")
plt.figure(figsize=(6.5, 4.2))
markers = {0: "o", 1: "x", 2: "s", 4: "^", 8: "D"}
for k in ks:
    W, N = (3, 1) if k == 0 else (4 * k + 1, k + 1)
    ls = sorted(l for (kk, l) in results if kk == k)
    fractions = [results[(k, l)][0] / results[(k, l)][1] for l in ls]
    l_hat = 0                                   # largest l such that all labellings were realised for every l' <= l
    for l, s in zip(ls, fractions):
        if s < 1.0:
            break
        l_hat = l
    marker = ">= " if l_hat == l_max else ""
    print(f"{k:3d} {W:4d} {marker + str(l_hat):>10} {2 * W * np.log2(np.e * N):15.1f}")
    plt.plot(ls, fractions, marker=markers.get(k, "o"), ls="--" if k == 0 else "-", ms=8 if k == 0 else 6,
             label=f"$k = {k}$ ($W = {W}$)")
plt.axhline(1, color="gray", lw=0.8)
plt.xlabel("number of points $l$"); plt.ylabel("fraction of realised labellings $s(l, k)$")
plt.legend(); plt.grid(alpha=0.3); plt.show()

########## YOUR SOLUTION HERE ##########